# Lab 3 – Clean the NYC Yellow Taxi data (January 2021)
INF4058 Data Engineering. Look → Profile → Find problems → Decide → Clean → Stop (no load to Postgres).

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

DATA = "data/yellow_tripdata_2021-01.csv.gz"
CHUNK = 100_000
DATE_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

# Same dtypes as ingest_data.py: nullable Int64 so a missing value does not turn the column into float
DTYPES = {
    "VendorID": "Int64",
    "passenger_count": "Int64",
    "trip_distance": "float64",
    "RatecodeID": "Int64",
    "store_and_fwd_flag": "string",
    "PULocationID": "Int64",
    "DOLocationID": "Int64",
    "payment_type": "Int64",
    "fare_amount": "float64",
    "extra": "float64",
    "mta_tax": "float64",
    "tip_amount": "float64",
    "tolls_amount": "float64",
    "improvement_surcharge": "float64",
    "total_amount": "float64",
    "congestion_surcharge": "float64",
}

## Step 1 – Look at the first 100 rows

In [2]:
df100 = pd.read_csv(DATA, nrows=100)
print(df100.head())
print()
print(df100.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1           2.10           1                  N           142            43             2          8.0    3.0      0.5   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1           0.20           1                  N           238           151             2          3.0    0.5      0.5   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1          14.70           1                  N           132           165             1         42.0    0.5      0.5   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0          10.60           1                  N           138           132             1         29.0    0.5      0.5   
4         2  2021-01-01 00:31:49   2021-

One row = one taxi trip. The two `tpep_*_datetime` columns are dates but pandas reads them as plain text (`object`/`str`), so they must be parsed. `store_and_fwd_flag` is text. No column is empty in the first 100 rows — the nulls only appear later in the file.

## Step 2 – Profile the whole file in chunks of 100,000 (read-only)

In [3]:
total_rows = 0
nulls = None
mins, maxs = {}, {}
n_chunks = 0

for chunk in pd.read_csv(DATA, dtype=DTYPES, parse_dates=DATE_COLS, chunksize=CHUNK):
    n_chunks += 1
    total_rows += len(chunk)
    c_nulls = chunk.isna().sum()
    nulls = c_nulls if nulls is None else nulls + c_nulls
    for col in chunk.columns:
        if col == "store_and_fwd_flag":
            continue
        lo, hi = chunk[col].min(), chunk[col].max()
        if pd.notna(lo):
            mins[col] = lo if col not in mins else min(mins[col], lo)
            maxs[col] = hi if col not in maxs else max(maxs[col], hi)

cols = list(chunk.columns)
profile = pd.DataFrame({
    "dtype": chunk.dtypes.astype(str),
    "nulls": nulls,
    "null_%": (nulls / total_rows * 100).round(2),
    "min": pd.Series(mins),
    "max": pd.Series(maxs),
}).reindex(cols)
print(f"chunks read: {n_chunks}   total rows: {total_rows:,}")
print(profile)

chunks read: 14   total rows: 1,369,765
                                dtype  nulls  null_%                  min                  max
VendorID                        Int64  98352    7.18                    1                    2
tpep_pickup_datetime   datetime64[us]      0    0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime  datetime64[us]      0    0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count                 Int64  98352    7.18                    0                    8
trip_distance                 float64      0    0.00                  0.0            263163.28
RatecodeID                      Int64  98352    7.18                    1                   99
store_and_fwd_flag             string  98352    7.18                  NaN                  NaN
PULocationID                    Int64      0    0.00                    1                  265
DOLocationID                    Int64      0    0.00                    1                  265
payment_ty

## Step 3 – Find problems (still read-only, still chunked)

In [4]:
jan_start = pd.Timestamp("2021-01-01")
feb_start = pd.Timestamp("2021-02-01")

counts = {
    "pickup outside Jan 2021": 0,
    "dropoff before pickup": 0,
    "dropoff > 1h before pickup": 0,
    "trip_distance > 100 miles": 0,
    "fare_amount < 0": 0,
    "fare_amount = 0": 0,
    "total_amount < 0": 0,
    "trip_distance < 0": 0,
    "trip_distance = 0": 0,
    "passenger_count = 0": 0,
    "passenger_count missing": 0,
}
row_hashes = []
outside_months = pd.Series(dtype="int64")

for chunk in pd.read_csv(DATA, dtype=DTYPES, parse_dates=DATE_COLS, chunksize=CHUNK):
    pu, do = chunk["tpep_pickup_datetime"], chunk["tpep_dropoff_datetime"]
    outside = (pu < jan_start) | (pu >= feb_start)
    counts["pickup outside Jan 2021"] += int(outside.sum())
    outside_months = outside_months.add(pu[outside].dt.to_period("M").astype(str).value_counts(), fill_value=0)
    counts["dropoff before pickup"] += int((do < pu).sum())
    counts["dropoff > 1h before pickup"] += int((do < pu - pd.Timedelta(hours=1)).sum())
    counts["trip_distance > 100 miles"] += int((chunk["trip_distance"] > 100).sum())
    counts["fare_amount < 0"] += int((chunk["fare_amount"] < 0).sum())
    counts["fare_amount = 0"] += int((chunk["fare_amount"] == 0).sum())
    counts["total_amount < 0"] += int((chunk["total_amount"] < 0).sum())
    counts["trip_distance < 0"] += int((chunk["trip_distance"] < 0).sum())
    counts["trip_distance = 0"] += int((chunk["trip_distance"] == 0).sum())
    counts["passenger_count = 0"] += int((chunk["passenger_count"] == 0).sum())
    counts["passenger_count missing"] += int(chunk["passenger_count"].isna().sum())
    # hash every row so duplicates can be found across chunks without holding the whole file
    row_hashes.append(pd.util.hash_pandas_object(chunk, index=False))

all_hashes = pd.concat(row_hashes, ignore_index=True)
counts["exact duplicate rows"] = int(all_hashes.duplicated().sum())

problems = pd.DataFrame({"rows": counts})
problems["% of file"] = (problems["rows"] / total_rows * 100).round(3)
print(problems)
print()
print("Pickups outside January 2021, by month:")
print(outside_months.astype(int).sort_index())

                             rows  % of file
pickup outside Jan 2021        24      0.002
dropoff before pickup        5642      0.412
dropoff > 1h before pickup      1      0.000
trip_distance > 100 miles      55      0.004
fare_amount < 0              6769      0.494
fare_amount = 0               642      0.047
total_amount < 0             6662      0.486
trip_distance < 0               0      0.000
trip_distance = 0           19952      1.457
passenger_count = 0         26726      1.951
passenger_count missing     98352      7.180
exact duplicate rows            0      0.000

Pickups outside January 2021, by month:
tpep_pickup_datetime
2008-12     1
2009-01     3
2020-12    16
2021-02     4
dtype: int64


In [5]:
# Where do the nulls come from? They are the same rows across several columns.
nulls_by_vendor = []
for chunk in pd.read_csv(DATA, dtype=DTYPES, parse_dates=DATE_COLS, chunksize=CHUNK):
    miss = chunk["VendorID"].isna()
    nulls_by_vendor.append(chunk.loc[miss, ["passenger_count", "RatecodeID", "store_and_fwd_flag", "payment_type"]].isna().all(axis=1))
s = pd.concat(nulls_by_vendor)
print(f"rows with VendorID missing: {len(s):,}")
print(f"... of which passenger_count, RatecodeID, store_and_fwd_flag and payment_type are ALL missing too: {int(s.sum()):,}")

# Who produces the 'dropoff before pickup' rows, and by how much?
neg = []
for chunk in pd.read_csv(DATA, dtype=DTYPES, parse_dates=DATE_COLS, chunksize=CHUNK):
    m = chunk["tpep_dropoff_datetime"] < chunk["tpep_pickup_datetime"]
    neg.append(pd.DataFrame({
        "vendor_missing": chunk.loc[m, "VendorID"].isna(),
        "seconds": (chunk.loc[m, "tpep_dropoff_datetime"] - chunk.loc[m, "tpep_pickup_datetime"]).dt.total_seconds(),
    }))
neg = pd.concat(neg)
print()
print("dropoff-before-pickup rows by source (VendorID missing?):")
print(neg["vendor_missing"].value_counts())
print()
print("how far before pickup (seconds):")
print(neg["seconds"].describe())

rows with VendorID missing: 98,352
... of which passenger_count, RatecodeID, store_and_fwd_flag and payment_type are ALL missing too: 98,352



dropoff-before-pickup rows by source (VendorID missing?):
vendor_missing
True     5640
False       2
Name: count, dtype: int64

how far before pickup (seconds):
count    5.642000e+03
mean    -1.464237e+03
std      1.079044e+05
min     -8.105078e+06
25%     -4.100000e+01
50%     -2.700000e+01
75%     -1.300000e+01
max     -1.000000e+00
Name: seconds, dtype: float64


## Step 4 – Decide

| Problem | Decision | Why | Who owns the rule |
|---|---|---|---|
| Exact duplicate rows | **Drop** (keep first) | Same trip recorded twice; it would double-count trips and revenue. | Data engineering |
| Pickup outside January 2021 | **Drop** | The file is the January 2021 extract; 2008/2009/Dec-2020/Feb-2021 dates are meter clock errors or late records that belong to another file. | Data engineering (agreed with TLC/data provider) |
| Dropoff more than 1 hour before pickup | **Drop** | Impossible and not a rounding issue (one drops off in Oct 2020); the timestamps are garbage. | Data engineering |
| Dropoff 0–60 min before pickup | **Flag** (`flag_negative_duration`) | Almost all come from the rows with no VendorID and are seconds off: a clock/recording issue in that source. Fare, distance and locations look real, so the trip is kept but its duration must not be used. | Data engineering + data provider |
| trip_distance > 100 miles | **Flag** (`flag_long_distance`) | Max is 263,163 miles — clearly an odometer error, but the fare can still be valid. | Analysts |
| Negative fare / total | **Drop** | These are refunds/voids, not trips. Finance can still read them from the raw file. | Business/finance analysts |
| Zero fare | **Flag** (`flag_zero_fare`) | Could be a real cancelled/no-charge trip; dropping would hide it. Analysts decide per question. | Business analysts |
| trip_distance = 0 | **Flag** (`flag_zero_distance`) | Often a cancelled trip or a GPS failure, but the fare can still be real. Keep it, mark it. | Analysts |
| passenger_count = 0 | **Fix** → set to missing (`<NA>`) | Driver-entered field; 0 passengers is not a real value, but the trip itself is fine. | Data engineering |
| Missing VendorID / passenger_count / RatecodeID / payment_type / store_and_fwd_flag | **Flag** (`flag_incomplete`) | Same ~98k rows miss all of them together (a different data source), but times, locations and amounts are present and usable. | Data engineering + data provider |

## Step 5 – Clean

In [6]:
def clean(df: pd.DataFrame):
    """Return (cleaned_df, {rule: rows_removed}). The input dataframe is never modified."""
    out = df.copy()
    removed = {}

    def drop(rule, mask):
        nonlocal out
        mask = mask.fillna(False)
        removed[rule] = int(mask.sum())
        out = out.loc[~mask]

    # --- drop rules (applied in this order; each count is what that rule removed) ---
    drop("exact duplicate rows", out.duplicated(keep="first"))
    pu, do = out["tpep_pickup_datetime"], out["tpep_dropoff_datetime"]
    drop("pickup outside Jan 2021", (pu < "2021-01-01") | (pu >= "2021-02-01"))
    drop("dropoff > 1h before pickup",
         out["tpep_dropoff_datetime"] < out["tpep_pickup_datetime"] - pd.Timedelta(hours=1))
    drop("negative fare or total", (out["fare_amount"] < 0) | (out["total_amount"] < 0))

    # --- fix rules (no rows removed) ---
    zero_pax = out["passenger_count"] == 0
    out.loc[zero_pax.fillna(False), "passenger_count"] = pd.NA
    removed["passenger_count 0 -> NA (fixed, not removed)"] = 0

    # --- flag rules (no rows removed) ---
    out["flag_zero_fare"] = out["fare_amount"] == 0
    out["flag_zero_distance"] = out["trip_distance"] == 0
    out["flag_incomplete"] = out["VendorID"].isna()
    out["flag_negative_duration"] = out["tpep_dropoff_datetime"] < out["tpep_pickup_datetime"]
    out["flag_long_distance"] = out["trip_distance"] > 100
    removed["flags added (not removed)"] = 0

    return out.reset_index(drop=True), removed

In [7]:
# 1.37M rows fit in memory; duplicates must be checked across the whole month, so the cleaning step uses the full frame.
raw = pd.read_csv(DATA, dtype=DTYPES, parse_dates=DATE_COLS)
before_shape = raw.shape

cleaned, removed = clean(raw)

print(f"rows before: {len(raw):,}")
print(f"rows after:  {len(cleaned):,}")
print(f"rows removed in total: {len(raw) - len(cleaned):,}")
print()
print(pd.DataFrame({"rows_removed": removed}))

rows before: 1,369,765
rows after:  1,362,971
rows removed in total: 6,794

                                              rows_removed
exact duplicate rows                                     0
pickup outside Jan 2021                                 24
dropoff > 1h before pickup                               1
negative fare or total                                6769
passenger_count 0 -> NA (fixed, not removed)             0
flags added (not removed)                                0


In [8]:
# Checks: the original was not changed, nothing was dropped silently, and the rules really hold now
assert raw.shape == before_shape, "original dataframe changed!"
assert len(raw) - len(cleaned) == sum(removed.values()), "rows dropped that no rule accounts for"
assert not cleaned.duplicated(subset=[c for c in cleaned.columns if not c.startswith("flag_")]).any()
assert cleaned["tpep_pickup_datetime"].between("2021-01-01", "2021-01-31 23:59:59").all()
assert (cleaned["tpep_dropoff_datetime"] >= cleaned["tpep_pickup_datetime"] - pd.Timedelta(hours=1)).all()
assert (cleaned["fare_amount"] >= 0).all() and (cleaned["total_amount"] >= 0).all()
assert not (cleaned["passenger_count"] == 0).any()

flags = pd.DataFrame({
    "rows flagged": cleaned[[c for c in cleaned.columns if c.startswith("flag_")]].sum(),
})
flags["% of cleaned"] = (flags["rows flagged"] / len(cleaned) * 100).round(2)
print("All checks passed.")
print()
print(flags)

All checks passed.

                        rows flagged  % of cleaned
flag_zero_fare                   641          0.05
flag_zero_distance             19309          1.42
flag_incomplete                98238          7.21
flag_negative_duration          5639          0.41
flag_long_distance                55          0.00


## Step 6 – Stop here
The cleaned dataframe is **not** loaded into Postgres. Next session we plug `clean()` into `ingest_data.py`.